In [1]:
import numpy as np
import tensorflow
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd


In [2]:
df=pd.read_csv(r"C:\Users\shahh\OneDrive\Desktop\Data Science And Machine Learning\practice\SmartPhonePrediction\train.csv")

In [3]:
df.drop(columns=["id"],inplace=True)

In [4]:
numerical_cols = [
    'daily_screen_time_hours',
    'social_media_hours',
    'gaming_hours',
    'work_study_hours',
    'sleep_hours',
    'notifications_per_day',
    'app_opens_per_day',
    'weekend_screen_time',
    'age'
]


In [5]:
df.dropna(thresh=8,inplace=True)

In [6]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler





scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[numerical_cols])

imputer = IterativeImputer(
    max_iter=10,
    random_state=42
)

imputed_data = imputer.fit_transform(scaled_data)

# Convert back to original scale
imputed_data = scaler.inverse_transform(imputed_data)

# Put values back into dataframe
df[numerical_cols] = imputed_data

c:\Users\shahh\OneDrive\Desktop\Data Science And Machine Learning\.venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [7]:
#Feature Engineering
df["entertainment_hours"] = (
    df["social_media_hours"] +
    df["gaming_hours"]
)


df["entertainment_ratio"] = (
    df["entertainment_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["social_media_ratio"] = (
    df["social_media_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["gaming_ratio"] = (
    df["gaming_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["work_study_ratio"] = (
    df["work_study_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["weekend_screen_ratio"] = (
    df["weekend_screen_time"] /
    (df["daily_screen_time_hours"] + 0.01)
)


df["weekend_screen_difference"] = (
    df["weekend_screen_time"] -
    df["daily_screen_time_hours"]
)


df["recreational_to_work_ratio"] = (
    df["entertainment_hours"] /
    (df["work_study_hours"] + 0.01)
)


df["social_gaming_interaction"] = (
    df["social_media_hours"] *
    df["gaming_hours"]
)


df["sleep_screen_ratio"] = (
    df["sleep_hours"] /
    (df["daily_screen_time_hours"] + 0.01)
)

In [8]:
categorical_cols = [
    "gender",
    "stress_level",
    "academic_work_impact"
    
]

In [9]:
df[categorical_cols] = df[categorical_cols].fillna("Unknown")

In [10]:
df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [11]:
X=df.drop("addicted_label",axis=1)
y=df['addicted_label']

In [12]:
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42,stratify=y)

In [14]:
scaler=StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cu126
CUDA available: True
CUDA version: 12.6
Device: cuda
GPU: NVIDIA GeForce MX350


In [16]:
import sys
print(sys.executable)


c:\Users\shahh\OneDrive\Desktop\Data Science And Machine Learning\.venv\Scripts\python.exe


In [20]:
X = X.to(device)
y = y.to(device)

AttributeError: 'DataFrame' object has no attribute 'to'

In [ ]:
model = model.to(device)

NameError: name 'model' is not defined

In [ ]:
model=keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16,activation='relu'),
    layers.Dense(8,activation='relu'),
    layers.Dense(1,activation='sigmoid')
])

In [ ]:
model.compile(
    loss=keras.losses.BinaryCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

In [ ]:

history=model.fit(
    X_train,y_train,
    validation_data=(X_test,y_test),
    batch_size=4,
    epochs=100,
    verbose=1
)